In [ ]:
import os
import random
import time
from copy import deepcopy

import optuna
import torch
from sklearn.metrics import r2_score
from torch_geometric.data import Data
from tqdm import tqdm

import barostat_parameters
from graph_utils import LJInteractionParams, prepare_traj
from ift import no_bootstrap_rollout_implicit, specialized_rollout_implicit
from itpo_weights import DatasetType, ITPOWeights, ModelType
from pressure import compute_stress_curve, compute_total_stress
from simulator_SA_cpu_test import Model as VelocityModel
from training_utils import freeze_normalizer
from utils import (
    build_velocity_graph_correction,
    calc_p_ratio_box_tensor,
    load_and_split_dataset,
    specialized_rollout,
    specialized_rollout_cascade,
    specialized_rollout_STE,
    to_f32,
    visualize_nu_disribution,
)


### Data

We search for ITPO weights using intermidiate ($\nu \in (0.1, 0.2)$) data.

To check the how well these weights perform, different validation/test is used.

In [ ]:
poisson_buckets = [
    {"max": 0.1, "count": 300},                # P < 0.1
    {"min": 0.1, "max": 0.2, "count": 100},    # 0.1 <= P < 0.2
    {"min": 0.2, "count": 100}                 # P >= 0.2
]

# poisson_buckets = [
#     {"min": 0.0, "max": 1.0, "count": 400},    # full range same bucket
# ]

dataset_type = DatasetType.NodeOptimized

train_files, val_files, test_files = load_and_split_dataset(
    # registry_path="./data_mini/data_registry_mini.csv",
    registry_path="./data/data_registry.csv",
    # registry_path="./data/data_LJ_noisy_eps0.01_sigma1.0_cutoff1.122/data_registry.csv",
    target_data_type=dataset_type,
    possion_buckets=poisson_buckets,
    split_ratios=(0.5, 0.25, 0.25),
    seed=42
)

# Load actual data
data = {
    'train': {},
    'val' : {},
    'test' : {},
}
max_sim_len = 300
# stride = 5
print("Loading data...")
for key in data:
    if key == 'train':
        data[key] = [torch.load(file, weights_only=False)[:max_sim_len] for file in tqdm(train_files, desc=f"{key:<5} data")]
        # data[key] = [torch.load(file, weights_only=False)[::stride] for file in tqdm(train_files, desc=f"{key:<5} data")]
    elif key == 'val':
        data[key] = [torch.load(file, weights_only=False)[:max_sim_len] for file in tqdm(val_files, desc=f"{key:<5} data")]
        # data[key] = [torch.load(file, weights_only=False)[::stride] for file in tqdm(val_files, desc=f"{key:<5} data")]
    elif key == 'test':
        data[key] = [torch.load(file, weights_only=False)[:max_sim_len] for file in tqdm(test_files, desc=f"{key:<5} data")]
        # data[key] = [torch.load(file, weights_only=False)[::stride] for file in tqdm(test_files, desc=f"{key:<5} data")]
    else:
        raise ValueError(f"Unexpected key in data dictionary: {key}. ")

print("\nPreparing data...")
for data_type, sims in data.items():
    prepared = []
    for sim in tqdm(sims, desc=f"{data_type:<5} data"):
        lj_params = LJInteractionParams(0.01, 1.0, 1.122) if dataset_type is DatasetType.LJNoisy else None
        prepared_sim = prepare_traj(sim, lj_params, calc_angles=False)
        prepared.append(prepared_sim)
    data[data_type] = prepared

print(f"\nTrain data: {len(data['train'])} sims.")
print(f"Val data:   {len(data['val'])} sims.")
print(f"Test data:  {len(data['test'])} sims.")

visualize_nu_disribution(data)


### Load pretrained models

In [ ]:
device = "cpu"
simulator_type = ModelType.GNNModel # ModelType.GNNModel | ModelType.SimulatorCascade

match simulator_type:
    case ModelType.GNNModel:
        mp_layers = 2
        mlp = 3
        hidden_size = 128
        history = 3

        edge_features = 7 if dataset_type is DatasetType.LJNoisy else 4
        init_graph = Data(x=torch.ones((100, history*2)), edge_attr=torch.ones((100, edge_features)))

        models = {}

        # model: VelocityModel = VelocityModel(init_graph, hidden_size, mp_layers, mlp).to(device)
        # model.load_checkpoint(f"./trained_models/{dataset_type}/OST/model_P>0.2_h{history}_nl{mp_layers}_mlp{mlp}_epochs100.pt")
        # model = freeze_normalizer(model)
        # models[f"h{history} P>0.2 ost"] = model

        # model: VelocityModel = VelocityModel(init_graph, hidden_size, mp_layers, mlp).to(device)
        # model.load_checkpoint(f"./trained_models/{dataset_type}/OST/model_P>0.1_h{history}_nl{mp_layers}_mlp{mlp}_epochs100.pt")
        # model = freeze_normalizer(model)
        # models[f"h{history} P>0.1 ost"] = model

        # model: VelocityModel = VelocityModel(init_graph, hidden_size, mp_layers, mlp).to(device)
        # model_path = f"./trained_models/{dataset_type}/MST/model_P>0.2_h{history}_nl{mp_layers}_mlp{mlp}_epochs100.pt"
        # model.load_checkpoint(model_path)
        # model = freeze_normalizer(model)
        # models[f"h{history} P>0.2 mst"] = model

        # model: VelocityModel = VelocityModel(init_graph, hidden_size, mp_layers, mlp).to(device)
        # model.load_checkpoint(f"./trained_models/{dataset_type}/MST/model_P>0.1_h{history}_nl{mp_layers}_mlp{mlp}_epochs100.pt")
        # model = freeze_normalizer(model)
        # models[f"h{history} P>0.1 mst"] = model

        model: VelocityModel = VelocityModel(init_graph, hidden_size, mp_layers, mlp).to(device)
        model_path = f"./new_trained_models/{dataset_type}/MST/checkpoint_epoch_120.pt"
        model.load_checkpoint(model_path)
        model = freeze_normalizer(model)
        models[f"h{history} P>0.1 mst"] = model

        # model: VelocityModel = VelocityModel(init_graph, hidden_size, mp_layers, mlp).to(device)
        # model_path = f"./new_trained_models/{dataset_type}/MST/checkpoint_epoch_20.pt"
        # model.load_checkpoint(model_path)
        # model = freeze_normalizer(model)
        # models[f"h{history} P>0.1 mst"] = model

        # model: VelocityModel = VelocityModel(init_graph, hidden_size, mp_layers, mlp).to(device)
        # model_path = "./LJ_trained_models/lj_noisy/MST/trained_model/checkpoint_epoch_87.pt"
        # model.load_checkpoint(model_path)
        # model = freeze_normalizer(model)
        # models[f"h{history} P>0.1 mst"] = model

        for i, model_name in enumerate(models.keys()):
            print(f"{i+1}. Model {model_name} from {model_path} .")

    case ModelType.SimulatorCascade:
        hidden_size = 128
        mp_layers = 2
        mlp = 3
        epochs = 100

        # model_save_path = os.path.join("./trained_models", f"{dataset_type}", "cascade", "refined_P>0.2")
        model_save_path = os.path.join("./full_range_models", f"{dataset_type}", "cascade", "refined")
        n_models = len(os.listdir(model_save_path))

        models = []
        for h in tqdm(range(n_models)):
            if h == 0:
                n_graphs = 1
            else:
                n_graphs = h+1
            
            init_graph = build_velocity_graph_correction([data['val'][0][i].cpu().detach() for i in range(n_graphs)], panic_at_positions=False).to(device)

            current_h_model = VelocityModel(init_graph, hidden_size, mp_layers, mlp).to(device)
            current_h_model.load_checkpoint(os.path.join(model_save_path, f"model_refined_h{h}_nl{mp_layers}_mlp{mlp}_epochs{epochs}.pt"))
            current_h_model = freeze_normalizer(current_h_model)
            for param in current_h_model.parameters():
                param.requires_grad = False
            models.append(current_h_model)

        print(f"Loaded cascade of {len(models)} pretrained models.")    


### Find best ITPO weights for GNN simulator model

In [ ]:
models.keys()

In [ ]:
ft_data = random.sample(data['train'], k=10)

[calc_p_ratio_box_tensor(s) for s in ft_data]

In [ ]:
model = models['h3 P>0.1 mst']
model = model.eval()

num_rollout_steps = 50

if dataset_type is DatasetType.NodeOptimized:
    barostat_config = barostat_parameters.node_optimizated 
elif dataset_type is DatasetType.StiffOptimized:
    barostat_config = barostat_parameters.stiff_optimized
elif dataset_type is DatasetType.StiffAngles:
    barostat_config = barostat_parameters.stiff_angles
elif dataset_type is DatasetType.Noisy:
    barostat_config = barostat_parameters.noisy
elif dataset_type is DatasetType.LJNoisy:
    barostat_config = barostat_parameters.lj_noisy
else:
    raise ValueError(f"Data type {dataset_type} does not have corresponding barostat parameters.")

def objective(trial: optuna.Trial) -> float:

    t_start = time.time()

    # Define hyperparameter search space
    # refinement_iterations = trial.suggest_int("refinement_iterations", 5, 30)
    refinement_iterations = 30
    
    lr = trial.suggest_float("lr", 1e-9, 1e-4, log=True)    
    lambda_force = trial.suggest_float("lambda_force", 1e-8, 1e-3, log=True)
    lambda_energy = trial.suggest_float("lambda_energy", 1e-8, 1e-3, log=True)
    lambda_pressure = trial.suggest_float("lambda_pressure", 1e-8, 1e-3, log=True)

    weights = ITPOWeights(
        refinement_iterations,
        lr,
        lambda_force,
        lambda_energy,
        lambda_pressure,
        pressure_tol=None,
        force_tol=None
    )

    # Collect results here
    results = {
        "gt_p": [],
        "pred_p": [],
        "r2": [],
        "pos_mse": [],
        "pxx_mse": [],
    }

    # Evaluation Loop
    # intermidiate_data = [sim for sim in data['train'] if calc_p_ratio_box_tensor(sim) >= 0.1 and calc_p_ratio_box_tensor(sim) < 0.2]
    # for sim in intermidiate_data[:5]:
    with torch.no_grad():
        for sim in ft_data:

            # fresh_clone_graph = deepcopy(sim[0].cpu().detach().clone())
            input_graphs = [sim[i].detach().cpu() for i in range(history+1)]
            
            sim_strain = (sim[1].box.x - sim[-1].box.x) / sim[0].box.x
            assumed_rollout_length = int(sim_strain / 1e-5 / 0.01)
            dump_period = int(assumed_rollout_length / len(sim)) + 1

            barostat_config_current = deepcopy(barostat_config)
            barostat_config_current["default_skip"] = dump_period

            gt_p = calc_p_ratio_box_tensor(sim[:num_rollout_steps])
            results["gt_p"].append(gt_p.item())

            # Rollout prediction
            rollout = no_bootstrap_rollout_implicit(
                # starting_graph=input_graphs[0],
                input_graphs=input_graphs,
                gnn_simulator=model,
                gnn_history=history,
                barostat_config=barostat_config_current,
                lj_params=lj_params,
                itpo_weights=weights,
                # md_steps=(history + 1) * dump_period,
                rollout_steps=num_rollout_steps,
                device=device
            )
            
            rollout = [to_f32(g) for g in rollout]

            # Poisson's ratio part of the score
            pred_p = calc_p_ratio_box_tensor(rollout)
            # Check for NaNs
            if torch.isnan(pred_p) or torch.isinf(pred_p):
                raise optuna.exceptions.TrialPruned("NaN/Inf detected")
            results["pred_p"].append(pred_p.item())

            # Position MSE part of the score
            # position_mse = torch.nn.functional.mse_loss(rollout[-1].x, sim[len(rollout)-1].x)
            # results["pos_mse"].append(position_mse.item())

            # Position MSE part of the score
            # r0=sim[0].edge_attr[:, -2]
            # gt_stress_curve = compute_stress_curve(sim[:len(rollout)], lj_cutoff=lj_params.cutoff, sample_stride=1, device=device)
            # pred_stress_curve = compute_stress_curve(rollout, lj_cutoff=lj_params.cutoff, sample_stride=1, device=device)
            # stress_curve_mse = torch.nn.functional.mse_loss(pred_stress_curve[:, 0], gt_stress_curve[:, 0])
            # assert len(gt_stress_curve) == len(pred_stress_curve)
            # results["pxx_mse"].append(stress_curve_mse.item())

            # Pruning
            # step_score = r2_so_far - pos_mse.item() - pxx_mse.item()
            # trial.report(step_score, step)
            
            # if trial.should_prune():
            #     raise optuna.exceptions.TrialPruned()

        # Calculate Objective (Maximize R^2)        
        r2 = r2_score(results["gt_p"], results["pred_p"])
        # mean_pos_mse = torch.tensor(results["pos_mse"]).mean().item()
        # mean_pxx_mse = torch.tensor(results["pxx_mse"]).mean().item()

    t_stop = time.time()
    print(f"R2: {r2:.3f} | {(t_stop - t_start):.2f} s.")
    # print(f"R2: {r2:.3f} | pos MSE: {mean_pos_mse:.5e} | stress MSE: {mean_pxx_mse:.5e} | {(t_stop - t_start):.2f} s.")
    final_score = r2
    # final_score = r2 - mean_pos_mse - mean_pxx_mse
    # final_score = 10e6 * mean_pxx_mse
    return final_score

# Create the study. We want to maximize the R2 score.
study = optuna.create_study(
    direction="maximize",
    # direction="minimize",
    pruner=optuna.pruners.MedianPruner()
)

print("Starting Optuna optimization...")
study.optimize(objective, n_trials=200, show_progress_bar=False)

print("Best Trial:")
trial = study.best_trial
print(f"  R2 Score: {trial.value}")
print("  Best Hyperparameters:")
for key, value in trial.params.items():
    print(f"    {key}: {value}")


### Find ITPO weights for simulator cascade

In [ ]:
# intermidiate_data = [sim for sim in data['train'] if calc_p_ratio_box_tensor(sim) >= 0.1 and calc_p_ratio_box_tensor(sim) < 0.2]
intermidiate_data = data['val']
factors = [(sim[4].box_tensor[0] - sim[3].box_tensor[0]).item() for sim in intermidiate_data]
mean_delta_x = sum(factors)/len(factors)

num_rollout_steps = 100
dump_period = 200

if dataset_type is DatasetType.NodeOptimized:
    barostat_config = barostat_parameters.node_optimizated
elif dataset_type is DatasetType.StiffOptimized:
    barostat_config = barostat_parameters.stiff_optimized
elif dataset_type is DatasetType.Noisy:
    barostat_config = barostat_parameters.noisy

def objective(trial: optuna.Trial) -> float:

    # Define hyperparameter search space
    # refinement_iterations = trial.suggest_int("refinement_iterations", 5, 30)
    refinement_iterations = 20
    lr = trial.suggest_float("lr", 1e-8, 1e-3, log=True)
    lambda_force = trial.suggest_float("lambda_force", 1e-8, 1.0, log=True)
    lambda_energy = trial.suggest_float("lambda_energy", 1e-8, 1.0, log=True)
    lambda_pressure = trial.suggest_float("lambda_pressure", 1e-8, 1.0, log=True)

    # Collect results
    gt_poissons = []
    pred_poissons = []
    
    # Evaluation Loop
    for sim in intermidiate_data[:5]:
        fresh_clone_graph = deepcopy(sim[0].cpu().detach().clone())
        input_graphs = [fresh_clone_graph, ]
        rollout = specialized_rollout_cascade(
            starting_graph=input_graphs[0],
            gnn_models=models,
            barostat_config=barostat_config,
            box_delta_x=mean_delta_x,
            itpo_weights=ITPOWeights(refinement_iterations, lr, lambda_force, lambda_energy, lambda_pressure),
            rollout_steps=num_rollout_steps,
            device="cuda",
        )


        pred_p = calc_p_ratio_box_tensor(rollout)
        
        # Check for NaNs (physics explosion)
        if torch.isnan(pred_p) or torch.isinf(pred_p):
            raise ValueError("Physics Divergence (NaN/Inf detected)")

        pred_poissons.append(pred_p.item())
        
        gt_p = calc_p_ratio_box_tensor(sim[:num_rollout_steps])
        gt_poissons.append(gt_p.item())

    if len(pred_poissons) < 2:
        return -100.0 # Safety check for R2 calculation
        
    score = r2_score(gt_poissons, pred_poissons)
    
    return score


study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner()
)

print("Starting Optuna optimization...")
study.optimize(objective, n_trials=300, show_progress_bar=False)

print("\nOptimization Finished!")
print("Best Trial:")
trial = study.best_trial
print(f"  R2 Score: {trial.value}")
print("  Best Hyperparameters:")
for key, value in trial.params.items():
    print(f"    {key}: {value}")
